## LANGCHAIN - LESSON 2: TOOLS & TOOL CALLING - LLM Contract (Core & Intermediate)

Welcome to Lesson 2! In the previous lesson, we learned how to build the **Agent Harness** using `create_agent`. However, our agent was isolated. It only knew what was in its training data.

In Lesson 1, we learned that the **Agent Harness** (`create_agent`) wraps around the LLM to provide structure, memory, and control. But an agent without **Tools** is like a brain without hands—it can think, but it cannot act on the world.

**Agent = Model + Harness + Tools**

A **Tool** is a Python function that the model can decide to call during its reasoning loop.

---

## What is a Tool in LangChain?
A critical concept to understand in modern AI architecture is this: **The LLM does NOT execute code.**

When an agent uses a tool, the following flow occurs:
1. The Harness sends a list of available tools (formatted as JSON schemas) to the LLM.
2. The LLM decides it needs a tool and responds with the **Tool Name** and the **Arguments**.
3. The Agent Harness pauses the LLM, executes the local Python function, and returns the result to the LLM.
4. The LLM reads the result and formulates a final human-readable answer.
---

### The Tool Calling Loop

When you give an agent access to tools, a powerful autonomous loop emerges:

1. **User sends a message** → The agent receives the request
2. **LLM decides** → The model analyzes if it needs external information or action
3. **Tool Call** → If needed, the LLM generates a structured tool call (name + arguments)
4. **Harness executes** → `create_agent` intercepts the call, runs your Python function, and captures the result
5. **Re-injection** → The tool's output is fed back to the LLM as a `ToolMessage`
6. **Final response** → The LLM synthesizes everything and responds to the user

This entire loop happens **automatically**. You don't write the orchestration code—LangChain's harness handles it.

---
### Why Tools Matter

Tools transform your agent from a passive chatbot into an **autonomous system** that can:
- Query databases or APIs
- Perform calculations
- Search the web
- Send emails
- Execute code
- Interact with any external system

---

### Why this is powerful
- The model never executes arbitrary code itself.
- You control exactly what the agent is allowed to do.
- Tools can access external systems (APIs, databases, files, etc.).

---
### Official way to create tools (2026)

**The Modern Way: The `@tool` Decorator**

LangChain provides multiple ways to define tools, but the **recommended standard in 2026** is the `@tool` decorator from `langchain.tools`.

It automatically:
- Extracts the function name
- Uses the **docstring** as the tool description (critical for the model)
- Builds the input schema from **type hints**

---
### Why `@tool`?

1. **Pythonic & Clean**: Just decorate any Python function
2. **Automatic Schema Generation**: LangChain extracts the function signature, types, and docstring to create a JSON Schema that the LLM understands
3. **Docstring = Instructions**: The docstring becomes the tool's description—this is **critical** because the LLM uses it to decide WHEN to call the tool
4. **Type Safety**: Pydantic validates arguments automatically

---
### The Golden Rule of Tool Design

> **Your docstring is your tool's "sales pitch" to the LLM.**
> If the docstring doesn't clearly explain what the tool does and when to use it, the agent will never call it.

---




# 1. Install dependencies

In [ ]:
# Install required packages.
# Note: Using `%pip` and `-U` ensures proper upgrades in the active Colab kernel.
%pip install -U langchain "langchain[google-genai]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 16.2 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.0
    Uninstalling langchain-core-1.6.0:
      Successfully uninstalled langchain-core-1.6.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled googl

# 2. Configuration and Imports

In [14]:
import os
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from google.colab import userdata

# 1. LLM API KEY
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')


# 3. Creating your First Tool

### Tool: `get_current_weather`
Notice the structure:
1. **Function name**: Clear and descriptive (`get_current_weather`)
2. **Type hints**: `city: str` tells the LLM what argument to pass
3. **Docstring**: Explains WHAT the tool does and WHEN to use it
4. **Return type**: `str` indicates what the tool returns

In [9]:
import json

# 2. Creating your First Tool
@tool
def get_current_weather(city: str) -> str:
    """
    Get the current weather conditions for a specific city.

    Use this tool when the user asks about:
    - Current weather conditions
    - Temperature, humidity, or wind speed
    - Whether it's raining, sunny, or cloudy

    Args:
        city: The name of the city to check weather for (e.g., "New York", "Tokyo")

    Returns:
        A string with current weather information including temperature, conditions, and humidity.
    """
    # Simulated weather data (in production, this would call a real API)
    weather_data = {
        "new york": "New York: 22°C (72°F), Partly Cloudy, Humidity: 65%",
        "london": "London: 15°C (59°F), Overcast, Humidity: 80%",
        "tokyo": "Tokyo: 28°C (82°F), Sunny, Humidity: 55%",
        "paris": "Paris: 18°C (64°F), Light Rain, Humidity: 75%",
        "sydney": "Sydney: 25°C (77°F), Clear Sky, Humidity: 60%"
    }

    # Normalize input and return weather data
    city_lower = city.lower()
    if city_lower in weather_data:
        return weather_data[city_lower]
    else:
        return f"Weather data not available for {city}. Please try another city."

# Let's inspect the tool's schema (what the LLM sees, what the decorator created)
print("Tool Name:", get_current_weather.name)
print("\nTool Description:")
print(get_current_weather.description)
print("\nTool Schema (JSON):")
print(json.dumps(get_current_weather.args_schema.model_json_schema(), indent=2))

Tool Name: get_current_weather

Tool Description:
Get the current weather conditions for a specific city.

Use this tool when the user asks about:
- Current weather conditions
- Temperature, humidity, or wind speed
- Whether it's raining, sunny, or cloudy

Args:
    city: The name of the city to check weather for (e.g., "New York", "Tokyo")

Returns:
    A string with current weather information including temperature, conditions, and humidity.

Tool Schema (JSON):
{
  "description": "Get the current weather conditions for a specific city.\n\nUse this tool when the user asks about:\n- Current weather conditions\n- Temperature, humidity, or wind speed\n- Whether it's raining, sunny, or cloudy\n\nArgs:\n    city: The name of the city to check weather for (e.g., \"New York\", \"Tokyo\")\n\nReturns:\n    A string with current weather information including temperature, conditions, and humidity.",
  "properties": {
    "city": {
      "title": "City",
      "type": "string"
    }
  },
  "requ

# Agent - Harness Integration

## Integrating Tools with `create_agent`

Now we pass our tool to the agent harness via the `tools` parameter. The harness automatically:
1. Converts the tool to a JSON Schema
2. Injects it into the system prompt
3. Sets up the tool calling loop
4. Handles execution and re-injection

**Important**: The `system_prompt` should guide the agent on HOW to use tools, but the tool's docstring tells it WHEN to use them.

In [15]:
# 3. Model (the motor)
model = init_chat_model(
    "google_genai:gemini-3.6-flash",
    temperature=0.2,          # Lower temperature → more deterministic tool use
)

# 4. Agent Harness with the tool
agent = create_agent(
    model=model,
    tools=[get_current_weather],      # ← This is the key line
        system_prompt="""You are a helpful weather assistant.
        When users ask about weather, use the get_current_weather tool to provide accurate information.
        Always respond in a friendly and concise manner.""",
)

# 5. Invoke
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the weather in New York?"}
    ]
})

# 6. Clean output
last_message = result["messages"][-1]
print("AI 🤖:", last_message.content[0].get('text', ''))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AI 🤖: The weather in New York is currently 22°C (72°F) and partly cloudy with a humidity of 65%.


### Running the Agent with Tool Calling

## Testing the Agent: First Tool Call

Let's ask the agent about the weather in Tokyo, New York, Paris, Must be any city within weather_data to find a match; otherwise, no response is given. Watch how the harness automatically:
1. Recognizes the need for weather data
2. Calls `get_current_weather` with the correct argument
3. Receives the tool's response
4. Synthesizes a final answer